In [23]:
# ============================================================
# 0. KURULUM
# ============================================================
AKIS = "2025"   # Orijinal kod private repodan kopyalanır

import os
REPO = "prediction_of_ACP_2025"
if os.path.basename(os.getcwd()) != REPO:
    if not os.path.isdir(REPO):
        !git clone -q https://github.com/OzgenOzan/prediction_of_ACP_2025.git
    os.chdir(REPO)
else:
    !git pull -q   # oturum içinde tekrar çalıştırmada yeni aktarılan dosyaları çek
print("Çalışma dizini:", os.getcwd())

gerekli = [
    "_dataset_raw & UCLUST/working/merged-main-working.csv",
    "_dataset_raw & UCLUST/working/merged-ind-working.csv",
    "_dataset_raw & UCLUST/working/prediction_set.csv",
]
eksik = [f for f in gerekli if not os.path.exists(f)]
if eksik:
    raise FileNotFoundError(
        "Eksik dosyalar: " + ", ".join(eksik) + "\n"
        "→ GitHub'da OzgenOzan/prediction_of_ACP_2025 sayfasındaki README '2. adım'ı izleyerek "
        "'Import 2025 files' iş akışını bir kez çalıştırın (dosyalar otomatik aktarılır), "
        "sonra bu hücreyi yeniden çalıştırın."
    )

import ast
import joblib
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             f1_score, roc_auc_score, recall_score)
import sklearn
print("scikit-learn:", sklearn.__version__, "| pandas:", pd.__version__)

DATA = "_dataset_raw & UCLUST/working"   # kümelenmiş + dengelenmiş 2025 verisi

Çalışma dizini: /content/prediction_of_ACP_2025
scikit-learn: 1.6.1 | pandas: 2.2.3


## 1. Sekans → sayısal kodlama

`1. aa to numeric & stats/aa to numeric.ipynb` akışı: 26 sembollük eşleme (standart dışı amino asitler dahil; bilinmeyen karakter → 25 = X). Üç veri setine de uygulanır: eğitim (`merged-main-working`), bağımsız doğrulama (`merged-ind-working`), tahmin (`prediction_set`, DBAASP kaynaklı ~10k antimikrobiyel peptit).

In [24]:
# ============================================================
# 1. SAYISAL KODLAMA (aa to numeric.ipynb akışı)
# ============================================================
amino_acid_numeric_map = {"A": 1, "R": 2, "N": 3, "D": 4, "C": 5, "Q": 6, "E": 7, "G": 8, "H": 9,
                          "I": 10, "L": 11, "K": 12, "M": 13, "F": 14, "P": 15, "O": 16, "S": 17,
                          "U": 18, "T": 19, "W": 20, "Y": 21, "V": 22, "B": 23, "Z": 24, "X": 25, "J": 26}

def convert_sequence_to_numeric(sequence, mapping):
    # Haritada olmayan karakter 25'e (X) dönüştürülür
    return [mapping.get(aa, 25) for aa in sequence]

def sayisal_kodla(girdi_csv, cikti_csv):
    df = pd.read_csv(girdi_csv)
    if "sequence" not in df.columns:   # güvence: sekans kolonunu bul
        aday = [c for c in df.columns if "seq" in c.lower()]
        df = df.rename(columns={aday[0] if aday else df.columns[0]: "sequence"})
    df_cleaned = df.dropna(subset=["sequence"]).copy()
    df_cleaned["sequence"] = df_cleaned["sequence"].str.upper()
    df_cleaned["sequence"] = df_cleaned["sequence"].astype(str)
    df_cleaned["numeric_sequence"] = df_cleaned["sequence"].apply(
        lambda x: convert_sequence_to_numeric(x, amino_acid_numeric_map))
    df_cleaned.to_csv(cikti_csv, index=False)
    return df_cleaned

main_num = sayisal_kodla(f"{DATA}/merged-main-working.csv", "merged-main-numeric.csv")
ind_num  = sayisal_kodla(f"{DATA}/merged-ind-working.csv",  "merged-ind-numeric.csv")
pred_num = sayisal_kodla(f"{DATA}/prediction_set.csv",      "prediction-set-numeric.csv")

print(f"eğitim: {len(main_num)} | bağımsız: {len(ind_num)} | tahmin seti: {len(pred_num)}")
print("eğitim sınıf dağılımı:", main_num["anticancer"].value_counts().to_dict())

eğitim: 500 | bağımsız: 284 | tahmin seti: 15264
eğitim sınıf dağılımı: {0: 250, 1: 250}


## 2. Özellik hesaplama — AAC + DPC (+ one-hot)

`2. feature selection/feature_selec.ipynb` akışı: her sekans için **AAC (26)** + **DPC (676)** özelliği; one-hot encoding yalnızca ana veri setine uygulanır

In [25]:
# ============================================================
# 2. ÖZELLİK HESAPLAMA (feature_selec.ipynb akışı)
# ============================================================
def string_to_numeric_list(string_list):
    return [int(item) for item in ast.literal_eval(string_list)]

def calculate_AAC(sequence_string):
    sequence = string_to_numeric_list(sequence_string)
    aa_counts = {aa: sequence.count(aa) for aa in range(1, 27)}
    total_aa = len(sequence)
    return {aa: (count / total_aa) * 100 for aa, count in aa_counts.items()}

def calculate_DPC(sequence_string):
    sequence = string_to_numeric_list(sequence_string)
    dipeptides = list(product(range(1, 27), repeat=2))
    dipeptide_counts = {dipeptide: 0 for dipeptide in dipeptides}
    for i in range(len(sequence) - 1):
        dipeptide = (sequence[i], sequence[i + 1])
        dipeptide_counts[dipeptide] += 1
    total_dipeptides = len(sequence) - 1
    return {dipeptide: (count / total_dipeptides) * 100 for dipeptide, count in dipeptide_counts.items()}

def one_hot_encoding(sequence_string, max_length=113):
    sequence = string_to_numeric_list(sequence_string)
    one_hot_encoded = []
    for aa in sequence:
        vector = [0] * 26
        vector[int(aa) - 1] = 1
        one_hot_encoded.append(vector)
    for _ in range(max_length - len(sequence)):
        one_hot_encoded.append([0] * 26)
    return one_hot_encoded

def ozellik_hesapla(girdi_csv, cikti_csv, one_hot=False):
    df = pd.read_csv(girdi_csv)
    df_cleaned = df.dropna(subset=["numeric_sequence"]).copy()
    df_cleaned["numeric_sequence"] = df_cleaned["numeric_sequence"].astype(str)
    df_cleaned["AAC"] = df_cleaned["numeric_sequence"].apply(calculate_AAC)
    df_cleaned["DPC"] = df_cleaned["numeric_sequence"].apply(calculate_DPC)
    if one_hot:
        df_cleaned["One_Hot_Encoded"] = df_cleaned["numeric_sequence"].apply(one_hot_encoding)
    df_cleaned.to_csv(cikti_csv, index=False)
    return df_cleaned

main_feat = ozellik_hesapla("merged-main-numeric.csv", "merged-main-features.csv", one_hot=True)
ind_feat  = ozellik_hesapla("merged-ind-numeric.csv",  "merged-ind-features.csv")
pred_feat = ozellik_hesapla("prediction-set-numeric.csv", "prediction-set-features.csv")
print("Özellik dosyaları yazıldı: merged-main / merged-ind / prediction-set (-features.csv)")

Özellik dosyaları yazıldı: merged-main / merged-ind / prediction-set (-features.csv)


## 3. Model karşılaştırması — RF / LR / SVM / KNN

`3. best model selection/selection with aac&dpc.ipynb` akışı: aynı train/test ayrımı (`random_state=42`) ve ölçekleme ile dört model karşılaştırılır

In [26]:
# ============================================================
# 3. MODEL KARŞILAŞTIRMASI (selection with aac&dpc.ipynb akışı)
# ============================================================
def hazirlik(features_csv, hedef="anticancer"):
    """Orijinal notebook'lardaki ortak veri hazırlığı: AAC/DPC sözlük kolonları genişletilir,
    sayısallaştırılır, NaN->0, %80/%20 ayrım, StandardScaler eğitim bölümüne fit edilir."""
    df = pd.read_csv(features_csv)
    df = df.drop([c for c in ["sequence", "numeric_sequence", "One_Hot_Encoded"] if c in df.columns], axis=1)
    for col in ["AAC", "DPC"]:
        col_df = df[col].apply(lambda x: pd.Series(ast.literal_eval(x)))

        col_df.columns = [f"{col}_{str(c)}" for c in col_df.columns]
        df = pd.concat([df.drop(col, axis=1), col_df], axis=1)
    df.columns = [str(c) for c in df.columns]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df.fillna(0, inplace=True)
    oz = [c for c in df.columns if c.startswith(("AAC_", "DPC_"))]   # yalnızca özellik kolonları
    X = df[oz]
    y = df[hedef]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    return X, y, X_train_s, X_test_s, y_train, y_test, scaler

def metrikler(model_adi, y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred)
    print(f"--- {model_adi} ---")
    print(f"Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"Sensitivity: {cm[1,1] / (cm[1,1] + cm[1,0]):.4f}")
    print(f"Specificity: {cm[0,0] / (cm[0,0] + cm[0,1]):.4f}")
    print(f"Precision:   {precision_score(y_true, y_pred):.4f}")
    print(f"F1 Score:    {f1_score(y_true, y_pred):.4f}")
    print(f"ROC AUC:     {roc_auc_score(y_true, y_proba):.4f}\n")

X_main, y_main, X_train, X_test, y_train, y_test, scaler_main = hazirlik("merged-main-features.csv")
print("Özellik sayısı:", X_main.shape[1], "(26 AAC + 676 DPC = 702 beklenir)\n")

for ad, mdl in [("Random Forest", RandomForestClassifier(random_state=42)),
                ("Logistic Regression", LogisticRegression()),
                ("SVM", SVC(probability=True)),
                ("KNN", KNeighborsClassifier())]:
    mdl.fit(X_train, y_train)
    metrikler(ad, y_test, mdl.predict(X_test), mdl.predict_proba(X_test)[:, 1])

Özellik sayısı: 702 (26 AAC + 676 DPC = 702 beklenir)

--- Random Forest ---
Accuracy:    0.7900
Sensitivity: 0.7778
Specificity: 0.8043
Precision:   0.8235
F1 Score:    0.8000
ROC AUC:     0.9167

--- Logistic Regression ---
Accuracy:    0.8000
Sensitivity: 0.7778
Specificity: 0.8261
Precision:   0.8400
F1 Score:    0.8077
ROC AUC:     0.8603

--- SVM ---
Accuracy:    0.8000
Sensitivity: 0.8148
Specificity: 0.7826
Precision:   0.8148
F1 Score:    0.8148
ROC AUC:     0.9155

--- KNN ---
Accuracy:    0.6600
Sensitivity: 0.8519
Specificity: 0.4348
Precision:   0.6389
F1 Score:    0.7302
ROC AUC:     0.7436



## 4. RF hiperparametre optimizasyonu

`4. RF_aac&dpc Hyperparameter/rf_hyper_selec.ipynb` akışı: 5-katlı tabakalı CV ile GridSearchCV — önce geniş (1296 aday), sonra daraltılmış (162 aday)

In [5]:
# ============================================================
# 4. HİPERPARAMETRE OPTİMİZASYONU (rf_hyper_selec.ipynb akışı)
# ============================================================
param_grid = {
    'n_estimators': [40, 50, 60],
    'max_features': ['sqrt', "log2"],
    'max_depth': [12, 18, 24],
    'min_samples_split': [5, 10, 15],
    'min_samples_leaf': [1, 2, 3],
    'criterion': ["gini", "entropy"],
    'bootstrap': [True],
    'oob_score': [True, False],
    'warm_start': [True, False]
}
rf = RandomForestClassifier(random_state=42)
stratified_kfold = StratifiedKFold(n_splits=5)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=stratified_kfold, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print("En İyi Parametreler:", grid_search.best_params_)
print("En İyi Skor:", grid_search.best_score_)

# Daraltılmış ızgara (ikinci tur)
param_grid = {
    'n_estimators': [49, 50, 51],
    'max_features': ['sqrt', "log2"],
    'max_depth': [17, 18, 19],
    'min_samples_split': [9, 10, 11],
    'min_samples_leaf': [1, 2, 3],
    'criterion': ["gini"],
    'bootstrap': [True],
    'oob_score': [True],
    'warm_start': [True]
}
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=stratified_kfold, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print("En İyi Parametreler (daraltılmış):", grid_search.best_params_)
print("En İyi Skor (daraltılmış):", grid_search.best_score_)

Fitting 5 folds for each of 1296 candidates, totalling 6480 fits
En İyi Parametreler: {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 18, 'max_features': 'log2', 'min_samples_leaf': 3, 'min_samples_split': 10, 'n_estimators': 40, 'oob_score': True, 'warm_start': True}
En İyi Skor: 0.8625
Fitting 5 folds for each of 162 candidates, totalling 810 fits
En İyi Parametreler (daraltılmış): {'bootstrap': True, 'criterion': 'gini', 'max_depth': 19, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 9, 'n_estimators': 49, 'oob_score': True, 'warm_start': True}
En İyi Skor (daraltılmış): 0.8699999999999999


## 5. Final model — Random Forest

`5. work model/best param.ipynb` akışı: 4. bölümdeki optimizasyonun sonucu olan parametrelerle final model eğitilir, kaydedilir ve 10-katlı CV ile değerlendirilir

In [27]:
# ============================================================
# 5. FİNAL MODEL (best param.ipynb akışı)
# ============================================================
best_params = {
    'bootstrap': True, 'criterion': 'gini', 'max_depth': 18, 'max_features': 'sqrt',
    'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 50,
    'oob_score': True, 'warm_start': True
}

optimized_rf_model = RandomForestClassifier(**best_params, random_state=42)
optimized_rf_model.fit(X_train, y_train)
optimized_rf_predictions = optimized_rf_model.predict(X_test)
metrikler("Final RF (test)", y_test, optimized_rf_predictions,
          optimized_rf_model.predict_proba(X_test)[:, 1])

# Model ve scaler kaydedilir (orijinal dosyanın üzerine yazılmaz)
joblib.dump(optimized_rf_model, "5. work model/RF_acc_dpc_bestModel_colab.joblib")
joblib.dump(scaler_main, "5. work model/scaler.joblib")
print("Kaydedildi: 5. work model/RF_acc_dpc_bestModel_colab.joblib + scaler.joblib")

# 10-katlı çapraz doğrulama (orijinal akıştaki gibi)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
all_scores = {k: [] for k in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'sensitivity', 'specificity']}
all_predictions, all_true_labels = [], []

optimized_rf_model.set_params(warm_start=False)
for train_index, test_index in cv.split(X_main, y_main):
    X_tr, X_te = X_main.iloc[train_index], X_main.iloc[test_index]
    y_tr, y_te = y_main.iloc[train_index], y_main.iloc[test_index]
    optimized_rf_model.fit(X_tr, y_tr)
    predictions = optimized_rf_model.predict(X_te)
    probabilities = optimized_rf_model.predict_proba(X_te)[:, 1]
    all_predictions.extend(predictions)
    all_true_labels.extend(y_te)
    cm = confusion_matrix(y_te, predictions)
    all_scores['accuracy'].append(accuracy_score(y_te, predictions))
    all_scores['precision'].append(precision_score(y_te, predictions))
    all_scores['recall'].append(recall_score(y_te, predictions))
    all_scores['f1_score'].append(f1_score(y_te, predictions))
    all_scores['roc_auc'].append(roc_auc_score(y_te, probabilities))
    all_scores['sensitivity'].append(recall_score(y_te, predictions))
    all_scores['specificity'].append(cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) != 0 else 0)

scores_df = pd.DataFrame(all_scores)
predictions_df = pd.DataFrame({'True Labels': all_true_labels, 'Predictions': all_predictions})
scores_df.to_excel("model_scores.xlsx", index=False)
predictions_df.to_excel("model_predictions.xlsx", index=False)
print("\n10-katlı CV ortalamaları:")
print(scores_df.mean().round(4))

# CV döngüsü modelin durumunu değiştirdi; temiz bir final model için sıfırdan eğit
optimized_rf_model = RandomForestClassifier(**{**best_params, 'warm_start': False}, random_state=42)
optimized_rf_model.fit(X_train, y_train)

--- Final RF (test) ---
Accuracy:    0.8600
Sensitivity: 0.8519
Specificity: 0.8696
Precision:   0.8846
F1 Score:    0.8679
ROC AUC:     0.9215

Kaydedildi: 5. work model/RF_acc_dpc_bestModel_colab.joblib + scaler.joblib

10-katlı CV ortalamaları:
accuracy       0.8480
precision      0.8693
recall         0.8240
f1_score       0.8420
roc_auc        0.9213
sensitivity    0.8240
specificity    0.8720
dtype: float64


RandomForestClassifier(max_depth=18, min_samples_leaf=2, min_samples_split=10,
                       n_estimators=50, oob_score=True, random_state=42)

## 6. Bağımsız set ile doğrulama

`6. validation with indSet/validation.ipynb` akışı: bağımsız set aynı hazırlıktan geçer, model ve metrikler hesaplanır, 10-katlı CV çalıştırılır

In [28]:
# ============================================================
# 6. BAĞIMSIZ SET DOĞRULAMASI (validation.ipynb akışı)
# ============================================================
ORIJINAL_MODELI_DENE = True

X_ind, y_ind, X_ind_train, X_ind_test, y_ind_train, y_ind_test, _ = hazirlik("merged-ind-features.csv")

predictions_ind = model_sec6.predict(X_ind_test)
metrikler("Bağımsız set", y_ind_test, predictions_ind, model_sec6.predict_proba(X_ind_test)[:, 1])

# 10-katlı çapraz doğrulama (bağımsız set üzerinde, orijinal akıştaki gibi)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
all_scores_ind = {k: [] for k in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'sensitivity', 'specificity']}
all_predictions_ind, all_true_ind = [], []

for train_index, test_index in cv.split(X_ind, y_ind):
    X_tr, X_te = X_ind.iloc[train_index], X_ind.iloc[test_index]
    y_tr, y_te = y_ind.iloc[train_index], y_ind.iloc[test_index]
    cv_model = RandomForestClassifier(**{**best_params, 'warm_start': False}, random_state=42)
    cv_model.fit(X_tr, y_tr)
    p = cv_model.predict(X_te)
    prob = cv_model.predict_proba(X_te)[:, 1]
    all_predictions_ind.extend(p)
    all_true_ind.extend(y_te)
    cm = confusion_matrix(y_te, p)
    all_scores_ind['accuracy'].append(accuracy_score(y_te, p))
    all_scores_ind['precision'].append(precision_score(y_te, p))
    all_scores_ind['recall'].append(recall_score(y_te, p))
    all_scores_ind['f1_score'].append(f1_score(y_te, p))
    all_scores_ind['roc_auc'].append(roc_auc_score(y_te, prob))
    all_scores_ind['sensitivity'].append(recall_score(y_te, p))
    all_scores_ind['specificity'].append(cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) != 0 else 0)

pd.DataFrame(all_scores_ind).to_excel("model_scores_indSet.xlsx", index=False)
pd.DataFrame({'True Labels': all_true_ind, 'Predictions': all_predictions_ind}).to_excel("model_predictions_indSet.xlsx", index=False)

--- Bağımsız set ---
Accuracy:    0.7719
Sensitivity: 0.8000
Specificity: 0.7407
Precision:   0.7742
F1 Score:    0.7869
ROC AUC:     0.8802



## 7. ~10.000 antimikrobiyel peptitte tahmin

`7. prediction/prediction.ipynb` → `deleteSeqWithX.ipynb` → `csv to fasta.ipynb` akışı.

Sonuçlar `predictions.csv`, `filtered_predictions.csv` (X'siz) ve `filtered_predictions.fasta` olarak kaydedilir.

In [29]:
# ============================================================
# 7. TAHMİN (prediction.ipynb akışı)
# ============================================================
df = pd.read_csv("prediction-set-features.csv")

# Gruplama işlemi yap ve ilk örneği koru
df = df.groupby("sequence").first().reset_index()

# Rastgele 10.000 satır seç (set daha küçükse tamamı alınır)
n = min(10000, len(df))
df_sampled = df.sample(n=n, random_state=42)
print(f"Tahmin yapılacak tekil dizi sayısı: {n}")

sequence_column = df_sampled["sequence"]
df_sampled = df_sampled.drop([c for c in ["sequence", "numeric_sequence", "One_Hot_Encoded"] if c in df_sampled.columns], axis=1)

# AAC ve DPC sütunlarını işle
aac_dpc_columns = pd.DataFrame()
for col in ["AAC", "DPC"]:
    col_df = df_sampled[col].apply(lambda x: pd.Series(ast.literal_eval(x)))
    col_df.columns = [f"{col}_{str(c)}" for c in col_df.columns]
    aac_dpc_columns = pd.concat([aac_dpc_columns, col_df], axis=1)
    df_sampled = pd.concat([df_sampled.drop(col, axis=1), col_df], axis=1)

# Yalnızca özellik kolonları, eğitimdeki sırayla
oz_kolonlar = list(X_main.columns)
for c in oz_kolonlar:
    if c not in df_sampled.columns:
        df_sampled[c] = 0
df_sampled = df_sampled[oz_kolonlar].apply(pd.to_numeric, errors="coerce")
df_sampled.fillna(0, inplace=True)

# Özellik ölçeklendirme — AKIS anahtarı
if AKIS == "2026":
    scaler_pred = joblib.load("5. work model/scaler.joblib")   # eğitim scaler'ı
    X_scaled = scaler_pred.transform(df_sampled)
    print("Ölçekleme: eğitim scaler'ı (2026 düzeltilmiş akış)")
else:
    scaler_pred = StandardScaler()
    X_scaled = scaler_pred.fit_transform(df_sampled)           # orijinal 2025 akışı
    print("Ölçekleme: tahmin verisine yeni scaler fit edildi (2025 orijinal akışı)")

# Tahmin — 5. bölümde kaydedilen temiz model kullanılır (6. bölümdeki CV model durumunu değiştirmişti)
optimized_rf_model = joblib.load("5. work model/RF_acc_dpc_bestModel_colab.joblib")
predictions = optimized_rf_model.predict(X_scaled)

predictions_df = pd.DataFrame(predictions, columns=["anticancer_predicted"])
df_combined = pd.concat([sequence_column.reset_index(drop=True), predictions_df,
                         aac_dpc_columns.reset_index(drop=True)], axis=1)
df_combined.to_csv("predictions.csv", index=False)
print("Tahmin dağılımı:", df_combined["anticancer_predicted"].value_counts().to_dict())

# deleteSeqWithX.ipynb akışı: 'X' içeren dizileri filtrele
filtered_df = df_combined[~df_combined["sequence"].str.contains('X')]
filtered_df.to_csv("filtered_predictions.csv", index=False)

# csv to fasta.ipynb akışı: FASTA çıktısı
with open("filtered_predictions.fasta", "w") as fasta_file:
    for index, row in filtered_df.iterrows():
        fasta_file.write(f">seq{index}\n{row['sequence']}\n")

Tahmin yapılacak tekil dizi sayısı: 10000
Ölçekleme: tahmin verisine yeni scaler fit edildi (2025 orijinal akışı)
Tahmin dağılımı: {1: 6305, 0: 3695}


## 8. Çözünürlük özellikleri — PepFun

`8. solubility/PepFun Compiler.ipynb` akışı: filtrelenmiş tahminler için net yük, molekül ağırlığı, logP ve hidrofobisite hesaplar (`peptide_properties.csv`).

In [21]:
# ============================================================
# 8. PEPFUN ÇÖZÜNÜRLÜK
# ============================================================
CALISTIR_PEPFUN = True

if CALISTIR_PEPFUN:
    try:
        import sys
        import pandas as pd

        sys.path.append("8. solubility")
        from pepfun import *  # noqa: F401,F403

        # Standart 20 amino asit
        VALID_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")

        df_sol = pd.read_csv("filtered_predictions.csv")

        # Boş değerleri çıkar, dizileri büyük harfe çevir
        sequences = (
            df_sol["sequence"]
            .dropna()
            .astype(str)
            .str.strip()
            .str.upper()
            .tolist()
        )

        results = []
        errors = []

        for seq in sequences:
            # Standart olmayan amino asitleri belirle
            invalid_residues = sorted(set(seq) - VALID_AMINO_ACIDS)

            if invalid_residues:
                errors.append({
                    "sequence": seq,
                    "error_type": "UnsupportedResidue",
                    "error_message": (
                        "Desteklenmeyen karakterler: "
                        + ", ".join(invalid_residues)
                    )
                })
                continue

            try:
                pep = peptide_sequence(seq)  # noqa: F405

                pep.compute_peptide_charges()
                pep.calculate_properties_from_mol()
                pep.calculate_properties_from_sequence()

                results.append({
                    "sequence": seq,
                    "charge": pep.netCharge,
                    "weight": pep.mol_weight,
                    "logP": pep.mol_logp,
                    "hydrophobicity": pep.avg_hydro
                })

            except Exception as seq_error:
                # Tek bir dizide hata olması tüm analizi durdurmaz
                errors.append({
                    "sequence": seq,
                    "error_type": type(seq_error).__name__,
                    "error_message": str(seq_error)[:500]
                })

        # Başarılı hesaplamaları kaydet
        results_df = pd.DataFrame(
            results,
            columns=[
                "sequence",
                "charge",
                "weight",
                "logP",
                "hydrophobicity"
            ]
        )

        results_df.to_csv(
            "peptide_properties.csv",
            index=False
        )

        # Hatalı veya desteklenmeyen dizileri kaydet
        errors_df = pd.DataFrame(
            errors,
            columns=[
                "sequence",
                "error_type",
                "error_message"
            ]
        )

        errors_df.to_csv(
            "peptide_properties_errors.csv",
            index=False
        )

    except Exception as e:
        # Hücre altında yalnızca kısa, kontrollü bir hata gösterir
        print(
            "PepFun analizi başlatılamadı:",
            type(e).__name__,
            str(e)[:300]
        )

## 9. Çıktıları indir

In [30]:
# ============================================================
# 9. ÇIKTILARI İNDİR
# ============================================================
try:
    from google.colab import files
    files.download("predictions.csv")
    files.download("filtered_predictions.csv")
    files.download("filtered_predictions.fasta")
    files.download("peptide_properties.csv")
    files.download("model_scores.xlsx")
except ImportError:
    import os
    print("Colab dışı ortam — dosyalar çalışma dizininde:", os.getcwd())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>